DH演習2026 · 教材 第53版 · 2026-r53 · 2026-11-25

<details><summary>教材の版と相談用情報</summary>

相談時は次の情報を伝えてください。

```text
DH演習2026 · 教材 第53版
教材ID: 2026-r53
教材日付: 2026-11-25
活動: place_identity
環境: Colab / Jupyter
ソース: 3bb1cdc74699c6df82735f72bff2ce617a0b4964
```

</details>

[同じ版の教材・変更点](https://nlp.lang.osaka-u.ac.jp/dh/practicum/editions.html)


# 第8回：同じ地名は、同じ場所を指すか

確認問題まで上から順にセルを実行してください（Shift+Enter）。値を変えたら、そのセルから下を再実行します。判断表の記入と保存は、授業の教材（marimo）で行います。

確認問題の後の追加課題は授業外・任意です。予測とコードの変更は自分で行い、エラーが出たらAIに説明を求められます。

資料：[新板大坂之図](https://image.oml.city.osaka.lg.jp/da/detail?tilcod=0000000021-OSK0186760)と[難波橋の絵葉書](https://image.oml.city.osaka.lg.jp/da/detail?tilcod=0000000021-OSK0157268)（大阪市立図書館、CC0）、[Wikidata](https://www.wikidata.org/)（CC0）

## 1. 古地図で橋名を探す

N1は1657年の原図を1980年に複製した地図で、東が上（北が左）です。次のセルは原画像（約25 MB）を一度だけ取得し、川の部分と橋名の部分を拡大して表示します。提供元の縮小画像では橋名を読めないため、原画像を使います。

In [ ]:
#@title 操作：古地図を取得して拡大する
from pathlib import Path

import requests
from IPython.display import display
from PIL import Image

url = "https://image.oml.city.osaka.lg.jp/da/download/?id=0000000021-OSK0186760&size=org&type=image&file=%2F%E5%86%99%E7%9C%9F%E3%83%BB%E7%B5%B5%E3%81%AF%E3%81%8C%E3%81%8D%E3%83%BB%E5%8F%A4%E6%96%87%E6%9B%B8%E3%83%BB%E5%9C%B0%E5%9B%B3%2Fr1002001.jpg"
n1_path = Path("N1.jpg")
if not n1_path.exists():
    try:
        response = requests.get(url, timeout=180)
        response.raise_for_status()
        n1_path.write_bytes(response.content)
    except requests.RequestException as error:
        print("古地図を取得できませんでした。時間をおいて、このセルを再実行してください。")
        print("詳細：", error)
if n1_path.exists():
    n1 = Image.open(n1_path)
    display(n1.crop((100, 2300, 2000, 3600)).reduce(2), n1.crop((950, 2670, 1250, 2920)))

## 2. 絵葉書の年と景観を確かめる

N2の目録の刊年は[1915]です。角括弧は目録が補った年を示します。ライオン像・路面電車・欄干を探してください。

In [ ]:
#@title 操作：絵葉書を取得する
url = "https://image.oml.city.osaka.lg.jp/da/download/?id=0000000021-OSK0157268&size=view&type=image&file=%2F%E5%86%99%E7%9C%9F%E3%83%BB%E7%B5%B5%E3%81%AF%E3%81%8C%E3%81%8D%E3%83%BB%E5%8F%A4%E6%96%87%E6%9B%B8%E3%83%BB%E5%9C%B0%E5%9B%B3%2Fc0150001.jpg"
try:
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    Path("N2.jpg").write_bytes(response.content)
except requests.RequestException as error:
    print("絵葉書を取得できませんでした。時間をおいて、このセルを再実行してください。")
    print("詳細：", error)
if Path("N2.jpg").exists():
    display(Image.open("N2.jpg"))

## 3. Wikidataで候補を検索する

次の取得セルは、`live`が`False`の間、2026年11月25日にWikidataで「難波橋」と「天神橋」を検索した上位5件を表示します。授業の教材（marimo）の同梱の検索結果と同じ候補です。`live`を`True`にすると、検索語をWikidataへ送り、今の結果を取得します。

表には、候補の名前・説明・検索語に一致した表記と、最初の座標を表示します。座標は、現在の橋などを代表する一点です。

In [ ]:
query = "難波橋"
live = False

In [ ]:
#@title 操作：候補の表を用意する（このセルは読まなくてかまいません）
import pandas as pd

API = "https://www.wikidata.org/w/api.php"
HEADERS = {"User-Agent": "DHEnshu2026Colab/1.0 (https://nlp.lang.osaka-u.ac.jp/dh/practicum/)"}
COLUMNS = ["id", "名前", "説明", "一致した表記", "緯度", "経度"]

# 2026-11-25にWikidataで検索した上位5件（marimo版の同梱の検索結果と同じ）
saved_searches: dict = {
    "難波橋": [
        {"id": "Q11659546", "名前": "難波橋", "説明": "大阪市の橋", "一致した表記": "難波橋", "緯度": 34.692694, "経度": 135.507139},
        {"id": "Q11614009", "名前": "船場", "説明": "大阪市の中心業務地区", "一致した表記": "難波橋筋", "緯度": 34.68388337114979, "経度": 135.50346952385993},
        {"id": "Q858429", "名前": "なにわ橋駅", "説明": "大阪府大阪市北区にある京阪電気鉄道の駅", "一致した表記": "難波橋站", "緯度": 34.692972, "経度": 135.505937},
    ],
    "天神橋": [
        {"id": "Q101792030", "名前": "天神橋", "説明": "日本の大阪府大阪市北区の町", "一致した表記": "天神橋", "緯度": None, "経度": None},
        {"id": "Q11443265", "名前": "天神橋", "説明": "大阪市の橋", "一致した表記": "天神橋", "緯度": 34.69136111, "経度": 135.51172222},
        {"id": "Q141950606", "名前": "天神橋", "説明": "", "一致した表記": "天神橋", "緯度": None, "経度": None},
        {"id": "Q17218475", "名前": "天神橋", "説明": "福島市にある橋", "一致した表記": "天神橋", "緯度": 37.74758333333333, "経度": 140.46755555555555},
        {"id": "Q141944289", "名前": "天神橋", "説明": "", "一致した表記": "天神橋", "緯度": None, "経度": None},
    ],
}


def ask_wikidata(params):
    response = requests.get(API, headers=HEADERS, params={**params, "format": "json"}, timeout=30)
    response.raise_for_status()
    data = response.json()
    if "error" in data:
        raise ValueError(data["error"].get("info", "Wikidataがエラーを返しました"))
    return data


def first_point(entity):
    claims = entity.get("claims") or {}
    points = claims.get("P625") or []
    if not points:
        return None, None
    value = points[0].get("mainsnak", {}).get("datavalue", {}).get("value", {})
    return value.get("latitude"), value.get("longitude")


def search_live(text):
    params = {"action": "wbsearchentities", "search": text, "language": "ja", "uselang": "ja", "type": "item", "limit": 5}
    found = ask_wikidata(params)["search"]
    if not found:
        return []
    ids = "|".join(item["id"] for item in found)
    entities = ask_wikidata({"action": "wbgetentities", "ids": ids, "props": "claims"})["entities"]
    rows = []
    for item in found:
        latitude, longitude = first_point(entities.get(item["id"], {}))
        rows.append({"id": item["id"], "名前": item.get("label", ""), "説明": item.get("description", ""),
                     "一致した表記": item.get("match", {}).get("text", ""), "緯度": latitude, "経度": longitude})
    return rows


candidate_rows = []
if live:
    try:
        candidate_rows = search_live(query)
        print("Wikidataで「" + query + "」を検索しました。候補：", len(candidate_rows), "件")
    except (requests.RequestException, ValueError, KeyError) as error:
        print("Wikidataから候補を取得できませんでした。liveをFalseに戻すと、保存した結果で続けられます。")
        print("詳細：", error)
elif query in saved_searches:
    candidate_rows = saved_searches[query]
    print("保存した「" + query + "」の検索結果（2026年11月25日取得）を表示します。")
else:
    print("保存した検索結果は「難波橋」と「天神橋」だけです。ほかの地名は、liveをTrueにして検索してください。")
candidates = pd.DataFrame(candidate_rows, columns=COLUMNS)
candidates

## 4. 候補を絞る条件を変える

この節は「難波橋」の候補で試します。`exclude_stations`を`True`にすると、名前に「駅」を含む候補を除きます。`require_bridge`を`True`にすると、説明に「橋」を含まない候補を除きます。

`candidates.to_dict("records")`は、候補の表の各行を、列名をキーとする辞書にしたリストです。`and`は二つの条件がともに成り立つときだけ真になり、`not in`は指定した文字を含まないかを調べます。`if`の条件が成り立つと`elif`は調べません。`kept`には名前と候補IDを並べて残すので、同じ名前の候補も区別できます。

一つずつ`True`にして実行し、外れた候補と理由を確かめてください。二つとも`True`にする前に、どの候補がどちらの理由で外れるかを予測してください。説明が空だったり「橋」の字がなかったりすると、`require_bridge`は橋の項目でも除きます。

In [ ]:
exclude_stations = False
require_bridge = False

kept = []
for candidate in candidates.to_dict("records"):
    name = candidate["名前"] + "（" + candidate["id"] + "）"
    if exclude_stations and "駅" in candidate["名前"]:
        print("除く：", name, "名前に「駅」")
    elif require_bridge and "橋" not in candidate["説明"]:
        print("除く：", name, "説明に「橋」がない")
    else:
        kept.append(name)
print("全候補：", len(candidates), "残った候補：", kept)

## 5. 判断の基準を確かめる

同じ読みでも別の対象を指し、違う名前でも同じ対象を指すことがあります。中之島公園の案内図の写真は、授業の教材（marimo）①にあります。

| 表記 | 指しているもの | 表に残す関係 |
| --- | --- | --- |
| 難波橋 | 川を渡る橋 | 橋の名前 |
| ライオン橋 | 同じ橋の愛称 | 橋の別名 |
| なにわ橋駅 | 京阪中之島線の駅 | 橋とは別の対象 |

残った候補が資料の場所を指すかは、marimo版③の判断表に、判断した資料と理由とともに記します。

## 確認問題

3節の`query`を「天神橋」に変えて、候補の表を表示してください。N3の[大阪市の市街地図](https://www.city.osaka.lg.jp/kensetsu/cmsfiles/contents/0000032/32065/shigaizu_1.jpg)では、難波橋の東に天神橋が描かれています。

この橋に当たる候補の`id`を`answer`の`"id"`に入れてください。`"clue"`には、その候補を採用するか、別の候補を却下する手がかりを、表の「説明」か「一致した表記」から一つ入れてください。

In [ ]:
answer = {"id": "", "clue": ""}

In [ ]:
#@title 答えを確かめる
import re
import unicodedata

others = {
    "Q11659546": "難波橋の候補IDです。N3で難波橋の東に描かれた天神橋の候補を選んでください",
    "Q17218475": "説明は「福島市にある橋」です。N3は大阪市の地図です",
    "Q101792030": "説明は「日本の大阪府大阪市北区の町」です。N3で選ぶのは、川を渡る橋です",
    "Q141950606": "説明が空欄の候補です。説明から大阪市の橋と確かめられる候補を選んでください",
    "Q141944289": "説明が空欄の候補です。説明から大阪市の橋と確かめられる候補を選んでください",
}
if not isinstance(answer, dict) or "id" not in answer or "clue" not in answer:
    print('結果：不合格（`answer`は {"id": "", "clue": ""} の形のまま、""の中に答えを入れてください）')
else:
    qid = unicodedata.normalize("NFKC", str(answer["id"])).strip().strip("「」`'\" ").upper()
    clue = unicodedata.normalize("NFKC", str(answer["clue"])).strip()
    if not re.fullmatch(r"Q[1-9][0-9]*", qid):
        print("結果：不合格（候補IDは「Q」で始まる番号です。表の`id`の列から写してください）")
    elif qid in others:
        print("結果：不合格（" + others[qid] + "）")
    elif qid != "Q11443265":
        print("結果：不合格（「天神橋」の検索結果にない候補IDです。表の`id`の列から写してください）")
    elif not clue:
        print("結果：不合格（候補を採用するか、別の候補を却下する手がかりを、説明か一致した表記から一つ`clue`に入れてください）")
    else:
        print("結果：合格")

## 授業外・任意：名前の代わりに分類で候補を選ぶ

Wikidataは、項目が何であるかを`wdt:P31`（分類）で、分類の上下関係を`wdt:P279`（上位の分類）で記録します。分類とその上下関係を定めた語彙をオントロジーと呼びます。下のセルでは、3節の候補の分類を表にし、`kind`（`Q12280`は橋）に当たる候補を選びます。

既定の表は、2026年11月25日にWikidataから取得した「難波橋」の候補の分類です。現在のデータで試すときは、3節の`live`を`True`にして、3節から再実行します。

まず表を表示し、下の選択セルを実行してください。「上位の分類」は各候補の分類から一段たどった先で、`follow_subclasses`を`True`にすると上位の分類が`kind`の候補も選ばれます。表の「上位の分類のID」から新たに加わる候補を予測してから`True`に変え、名前に「橋」を含むのに分類では選ばれない候補を一つ挙げてください。一つの候補に分類や上位の分類が複数あると表に複数の行が出るため、`name not in selected`で同じ候補を二度加えないようにします。

分類はWikidataの編集者の判断です。資料の場所との同定は、授業の教材（marimo）③の判断表に、判断した資料と理由を書いて行います。

In [ ]:
#@title 操作：分類の表を用意する
saved = [
    ("難波橋", "Q158438", "アーチ橋", "Q12280", "橋"),
    ("難波橋", "Q537127", "道路橋", "Q12280", "橋"),
    ("船場", "Q738570", "中心業務地区", "Q123705", "都市の地区"),
    ("船場", "Q738570", "中心業務地区", "Q1133961", "商業地域"),
    ("船場", "Q738570", "中心業務地区", "Q28692737", "オフィス街"),
    ("船場", "Q738570", "中心業務地区", "Q106860125", "center"),
    ("なにわ橋駅", "Q55488", "鉄道駅", "Q548662", "停車場"),
    ("なにわ橋駅", "Q55488", "鉄道駅", "Q800279", "鉄道施設"),
    ("なにわ橋駅", "Q55488", "鉄道駅", "Q97987466", "operating control point"),
    ("なにわ橋駅", "Q55488", "鉄道駅", "Q124673697", "rail transport station"),
    ("なにわ橋駅", "Q22808403", "地下駅", "Q863404", "地下"),
    ("なにわ橋駅", "Q22808403", "地下駅", "Q12819564", "駅"),
]
columns = ["名前", "分類のID", "分類", "上位の分類のID", "上位の分類"]
if live:
    names = candidates.set_index("id")["名前"].to_dict()
    sparql = "SELECT ?item ?class ?classLabel ?parent ?parentLabel WHERE { VALUES ?item { " + " ".join("wd:" + i for i in names) + " } ?item wdt:P31 ?class . OPTIONAL { ?class wdt:P279 ?parent . } SERVICE wikibase:label { bd:serviceParam wikibase:language 'ja,en'. } }"
    response = requests.get("https://query.wikidata.org/sparql", params={"query": sparql, "format": "json"}, headers=HEADERS, timeout=60)
    response.raise_for_status()
    class_rows = []
    for row in response.json()["results"]["bindings"]:
        value = {key: row.get(key, {}).get("value", "") for key in ("item", "class", "classLabel", "parent", "parentLabel")}
        class_rows.append({
            "名前": names[value["item"].rsplit("/", 1)[-1]],
            "分類のID": value["class"].rsplit("/", 1)[-1],
            "分類": value["classLabel"],
            "上位の分類のID": value["parent"].rsplit("/", 1)[-1],
            "上位の分類": value["parentLabel"],
        })
else:
    class_rows = [{"名前": n, "分類のID": c, "分類": cl, "上位の分類のID": pid, "上位の分類": pl} for n, c, cl, pid, pl in saved]
pd.DataFrame(class_rows, columns=columns)

In [ ]:
kind = "Q12280"
follow_subclasses = False

named = []
selected = []
for row in class_rows:
    name = row["名前"]
    if "橋" in name and name not in named:
        named.append(name)
    is_direct = row["分類のID"] == kind
    is_parent = follow_subclasses and row["上位の分類のID"] == kind
    if (is_direct or is_parent) and name not in selected:
        selected.append(name)
print("名前に「橋」を含む候補：", named)
print("分類で選んだ候補：", selected)